# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities

# Compute cosine similarity for each sentence pair
for sentence1, sentence2 in sentence_pairs:

    # Convert the two sentences into embedding vectors
    embeddings = model.encode([sentence1, sentence2])

    # Compare the vectors using cosine similarity
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    print(f"Sentence 1: {sentence1}")
    print(f"Sentence 2: {sentence2}")
    print(f"Cosine similarity: {similarity:.3f}")
    print("-" * 60)

/opt/anaconda3/envs/dft/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 19:28:37.287000 50482 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9912.20it/s]


Sentence 1: A dog is playing in the park.
Sentence 2: A dog is running in a field.
Cosine similarity: 0.522
------------------------------------------------------------
Sentence 1: I love pizza.
Sentence 2: I enjoy ice cream.
Cosine similarity: 0.528
------------------------------------------------------------
Sentence 1: What is AI?
Sentence 2: How does a computer learn?
Cosine similarity: 0.319
------------------------------------------------------------


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


### Answer

- The pair “I love pizza” and “I enjoy ice cream” has the highest cosine similarity (0.528), followed closely by the two dog sentences (0.522). The food sentences share a similar structure, positive sentiment, and food-related meaning. The dog sentences also describe similar subjects and activities. The AI/computer-learning pair is related conceptually, but its similarity is lower (0.319).

- Cosine similarity depends on the quality of the embeddings. It may struggle with negation, sarcasm, ambiguous words, very domain-specific language, or sentences that use different wording to express the same meaning.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [2]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents
# Convert each document into an embedding vector
document_embeddings = model.encode(documents)

# Check the result
print("Number of documents:", len(documents))
print("Embedding matrix shape:", document_embeddings.shape)

Number of documents: 6
Embedding matrix shape: (6, 384)


In [3]:
# Perform KMeans clustering

# Group the document embeddings into 3 clusters
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(document_embeddings)

print("Cluster labels:", cluster_labels)

Cluster labels: [1 1 1 0 2 0]


In [4]:
# Print cluster assignments
# Display documents grouped by cluster
for cluster in sorted(set(cluster_labels)):
    print(f"\n--- Cluster {cluster} ---")

    for document, label in zip(documents, cluster_labels):
        if label == cluster:
            print(document)


--- Cluster 0 ---
How do I change a flat tire on a car?
How do I fix a leaky faucet?

--- Cluster 1 ---
What is the capital of France?
How do I bake a chocolate cake?
What is the distance between Earth and Mars?

--- Cluster 2 ---
What is the best way to learn Python?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

### Answer

- Three clusters are a reasonable starting point, but the result is not perfectly separated. Cluster 0 is quite coherent, while Cluster 1 contains several different topics. This suggests that three clusters may not be the optimal choice for such a small and diverse dataset.

- Cluster 0 can be described as practical repair/how-to tasks, because it contains changing a flat tire and fixing a leaky faucet. Cluster 2 contains programming/learning, represented by learning Python. Cluster 1 is less coherent because it mixes geography, cooking, and astronomy. This shows that KMeans will always assign every document to a cluster even when the semantic grouping is not very strong.

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [5]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings

# Convert all documents into embedding vectors
doc_embeddings = model.encode(documents)

# Check the result
print("Number of documents:", len(documents))
print("Embedding matrix shape:", doc_embeddings.shape)

Number of documents: 10
Embedding matrix shape: (10, 384)


In [6]:
# Create the search function
# This function encodes the user query and returns the top N most similar documents

def semantic_search(query, documents, doc_embeddings, top_n=5):

    # Encode the user's query
    query_embedding = model.encode([query])

    # Compare the query with all document embeddings
    similarities = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    # Get indices of the highest similarity scores
    top_indices = np.argsort(similarities)[::-1][:top_n]

    # Store the top matching documents and their scores
    results = []

    for index in top_indices:
        results.append({
            "document": documents[index],
            "similarity": similarities[index]
        })

    return results

In [7]:
# Test the search function
query = "Explain programming languages."

results = semantic_search(
    query,
    documents,
    doc_embeddings
)

print(f"Query: {query}\n")

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['document']} "
        f"(similarity: {result['similarity']:.3f})"
    )

Query: Explain programming languages.

1. What is quantum computing? (similarity: 0.435)
2. What is the best way to learn Python? (similarity: 0.319)
3. How do I build a mobile app? (similarity: 0.110)
4. How do I set up a local server? (similarity: 0.091)
5. What are the best travel destinations in Europe? (similarity: 0.091)


### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

### Answers

- The highest-ranked result was “What is quantum computing?”, followed by “What is the best way to learn Python?”. Although Python appears more directly related to programming languages, the embedding model ranked quantum computing higher, showing that semantic similarity is not always perfectly aligned with human judgement.

- The system could display similarity scores, highlight why a result may be related, use a stronger embedding model, or combine semantic similarity with keyword matching.